# Звук и спектрограмма руками

Вопрос лаборатории: какие сведения меняются на пути от звуковой волны до log-mel-спектрограммы? Мы вручную построим DFT и STFT, восстановим волну с фазой, отбросим фазу, создадим алиасинг и уничтожим противофазное стерео. Всё выполняется на CPU только с NumPy.

In [ ]:
import hashlib
import json
from pathlib import Path

import numpy as np

rng = np.random.default_rng(23)
print('NumPy:', np.__version__)


## 1. Физическое время и отсчёты

Частота дискретизации входит в смысл массива. Создадим четыре секунды звука с компонентами 8 и 12 Гц при 64 отсчётах в секунду.

In [ ]:
sample_rate = 64
duration_s = 4
time = np.arange(sample_rate * duration_s) / sample_rate
wave = 0.8 * np.sin(2 * np.pi * 8 * time) + 0.35 * np.sin(2 * np.pi * 12 * time)

assert wave.shape == (256,)
assert np.isclose(time[-1] + 1 / sample_rate, duration_s)
print('shape:', wave.shape, 'duration:', len(wave) / sample_rate, 's')


## 2. Один спектр без готового преобразования

Сначала умножаем окно на комплексные синусы вручную. Затем сравниваем результат с numpy.fft.rfft.

In [ ]:
frame_size = 64
window = np.hanning(frame_size)
frame = wave[:frame_size] * window
n = np.arange(frame_size)[None, :]
k = np.arange(frame_size // 2 + 1)[:, None]
fourier_matrix = np.exp(-2j * np.pi * k * n / frame_size)
manual_spectrum = fourier_matrix @ frame
library_spectrum = np.fft.rfft(frame)

np.testing.assert_allclose(manual_spectrum, library_spectrum, atol=1e-10, rtol=0)
frequency_hz = np.arange(len(manual_spectrum)) * sample_rate / frame_size
selected_bins = []
for index in np.argsort(np.abs(manual_spectrum))[::-1]:
    if all(abs(index - chosen) > 1 for chosen in selected_bins):
        selected_bins.append(int(index))
    if len(selected_bins) == 2:
        break
peaks = frequency_hz[selected_bins]
np.testing.assert_allclose(np.sort(peaks), [8.0, 12.0])
print('strongest frequencies:', np.sort(peaks), 'Hz')


## 3. STFT и точное восстановление

Режем сигнал на перекрывающиеся окна. При обратном проходе складываем окна и делим на сумму квадратов оконной функции. Комплексный спектр сохраняет и амплитуду, и фазу.

In [ ]:
def stft(signal, frame_size=64, hop=16):
    pad = frame_size // 2
    padded = np.pad(signal, (pad, pad))
    count = 1 + (len(padded) - frame_size) // hop
    frames = np.stack([
        padded[i * hop:i * hop + frame_size]
        for i in range(count)
    ])
    window = np.hanning(frame_size)
    return np.fft.rfft(frames * window[None, :], axis=-1), len(signal)

def istft(spectrum, original_length, frame_size=64, hop=16):
    window = np.hanning(frame_size)
    total = (len(spectrum) - 1) * hop + frame_size
    signal = np.zeros(total)
    weight = np.zeros(total)
    for index, row in enumerate(spectrum):
        start = index * hop
        frame = np.fft.irfft(row, n=frame_size)
        signal[start:start + frame_size] += frame * window
        weight[start:start + frame_size] += window ** 2
    valid = weight > 1e-12
    signal[valid] /= weight[valid]
    pad = frame_size // 2
    return signal[pad:pad + original_length]

complex_stft, original_length = stft(wave)
reconstructed = istft(complex_stft, original_length)
max_error = np.max(np.abs(reconstructed - wave))
assert complex_stft.shape == (17, 33)
assert max_error < 1e-10
print('STFT shape:', complex_stft.shape, 'max reconstruction error:', max_error)


## 4. Спектрограмма без фазы уже необратима

Из одинаковой амплитуды можно получить много волн с разной фазой. Подставим нулевую фазу и измерим ошибку.

In [ ]:
magnitude = np.abs(complex_stft)
power = magnitude ** 2
zero_phase_stft = magnitude.astype(np.complex128)
wrong_wave = istft(zero_phase_stft, original_length)
relative_phase_error = np.linalg.norm(wrong_wave - wave) / np.linalg.norm(wave)

assert relative_phase_error > 0.5
print('relative error after discarding phase:', relative_phase_error)

with np.errstate(divide='ignore'):
    broken_log = np.log(np.zeros((2, 2)))
assert not np.isfinite(broken_log).all()
safe_log_power = np.log(power + 1e-10)
assert np.isfinite(safe_log_power).all()


## 5. Mel-фильтры руками

Сожмём 33 линейные частоты в 8 полос. Границы и число полос становятся частью контракта модели.

In [ ]:
def hz_to_mel(hz):
    return 2595.0 * np.log10(1.0 + hz / 700.0)

def mel_to_hz(mel):
    return 700.0 * (10.0 ** (mel / 2595.0) - 1.0)

def mel_filterbank(sample_rate, frame_size, bands):
    max_mel = hz_to_mel(sample_rate / 2)
    points_hz = mel_to_hz(np.linspace(0, max_mel, bands + 2))
    bins = np.floor((frame_size + 1) * points_hz / sample_rate).astype(int)
    bins = np.clip(bins, 0, frame_size // 2)
    bank = np.zeros((bands, frame_size // 2 + 1))
    for band in range(1, bands + 1):
        left, center, right = bins[band - 1:band + 2]
        for index in range(left, center):
            bank[band - 1, index] = (index - left) / max(center - left, 1)
        for index in range(center, right):
            bank[band - 1, index] = (right - index) / max(right - center, 1)
    return bank

filterbank = mel_filterbank(sample_rate, frame_size, bands=8)
mel_power = power @ filterbank.T
log_mel = np.log(mel_power + 1e-10)
assert filterbank.shape == (8, 33)
assert log_mel.shape == (17, 8)
assert np.all(filterbank.sum(axis=1) > 0)
print('linear power:', power.shape, 'log-mel:', log_mel.shape)


## 6. Разрешение по времени и частоте

Окно не может одновременно быть очень коротким во времени и давать очень мелкую частотную сетку.

In [ ]:
def resolution(sample_rate, frame_size, hop):
    return {'frequency_hz': sample_rate / frame_size, 'time_s': hop / sample_rate}

short_window = resolution(sample_rate, frame_size=16, hop=4)
long_window = resolution(sample_rate, frame_size=64, hop=16)
assert short_window['frequency_hz'] > long_window['frequency_hz']
assert short_window['time_s'] < long_window['time_s']
print('short window:', short_window)
print('long window:', long_window)


## 7. Две намеренные поломки: алиасинг и mono

Сначала уменьшим частоту дискретизации без фильтра. Затем усредним два противофазных канала.

In [ ]:
tone_20_hz = np.sin(2 * np.pi * 20 * time)
downsampled = tone_20_hz[::2]
new_rate = sample_rate // 2
freqs = np.fft.rfftfreq(len(downsampled), d=1 / new_rate)
aliased_peak = freqs[np.argmax(np.abs(np.fft.rfft(downsampled)))]
assert np.isclose(aliased_peak, 12.0)

left = np.sin(2 * np.pi * 8 * time)
right = -left
stereo = np.stack([left, right], axis=-1)
mono = stereo.mean(axis=-1)
assert np.max(np.abs(mono)) < 1e-12
print('20 Hz became', aliased_peak, 'Hz after unsafe downsampling')
print('stereo energy:', np.mean(stereo ** 2), 'mono energy:', np.mean(mono ** 2))


## 8. Паспорт и дочерний артефакт

Параметры STFT нельзя хранить в голове автора: они должны версионироваться вместе с портом.

In [ ]:
wave_sha256 = hashlib.sha256(wave.tobytes()).hexdigest()
manifest = {
    'id': 'booklm-audio-front-end-spec-v1',
    # спецификацию портов ноутбук не читает, поэтому её SHA-256 не записан: None, а не выдуманная сумма
    'parents': {'booklm-multimodal-ports-spec-v1': None},
    'created_by': 'notebooks/06_audio_spectrogram_by_hand.ipynb',
    'physical_quantity': 'relative microphone amplitude',
    'sample_rate_hz': sample_rate,
    'channels': 1,
    'frame_size': frame_size,
    'hop': 16,
    'window': 'hann',
    'spectrum': 'rfft_power',
    'mel_bands': 8,
    'log_epsilon': 1e-10,
    'wave_sha256': wave_sha256,
    'invariants': {
        'complex_stft_reconstruction_max_error': float(max_error),
        'zero_phase_relative_error': float(relative_phase_error),
        'unsafe_downsampling_alias_hz': float(aliased_peak)
    }
}
manifest_path = Path('/tmp/booklm-audio-front-end-spec-v1.json')
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
assert json.loads(manifest_path.read_text(encoding='utf-8'))['wave_sha256'] == wave_sha256
print(manifest_path)


## 9. Связь с промышленной системой

complex_stft соответствует обратимому частотно-временному представлению; power — спектрограмме без фазы; filterbank — фиксированной mel-предобработке; log_mel — входу аудиокодировщика; manifest — версии порта. В промышленной модели меняются размеры, фильтры и кодировщик, но не необходимость хранить частоту, каналы, окно, шаг, маску и правило нормировки.

**Задание:** добавьте противоалиасинговый фильтр перед уменьшением частоты дискретизации и докажите, что ложный пик 12 Гц исчез. Затем поменяйте шаг STFT, пересоберите манифест и объясните, почему старые веса входного соединителя больше не относятся к тому же контракту.